In [42]:
# Включаем автоматическую перезагрузку внешних модулей
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
# Импортируем нашу функцию из созданного файла utils.py
import sys
import os

# Добавляем родительскую папку
sys.path.append(os.path.abspath('..'))

from src.improvements import (remove_zero_dimensions, 
                              encode_categories, 
                              evaluate_custom_model,
                              remove_outliers,
                              add_volume_feature)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
df_raw = pd.read_csv('../data/raw/diamonds.csv')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

base_lr = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
robust_lr = Pipeline([('scaler', RobustScaler()), ('model', LinearRegression())])
base_rf = Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))])

In [32]:
df_raw = encode_categories(df_raw)

df_raw = remove_zero_dimensions(df_raw)
# Выбираем фичи и таргет без логарифма
X1 = df_raw[['carat', 'depth', 'table', 'x', 'y', 'z', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y1 = df_raw['price']

# Вызываем функцию
evaluate_custom_model(X1, y1, base_lr, is_target_logged=False)

R²: 0.9100
CV R²: 0.8875 (+/- 0.0355)
RMSE: 1201.39
MAE: 790.37
MAPE: 42.99%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
          carat   +5119.2634
clarity_encoded    +823.9814
  color_encoded    +550.0614
              y    +153.9515
    cut_encoded    +143.0037
          depth     -22.6375
          table     -55.3208
              x    -387.4342
              z    -757.2520


In [35]:
# Применяем следующий шаг чистки
df_step2 = remove_outliers(df_raw)

X2 = df_step2[['carat', 'depth', 'table', 'x', 'y', 'z', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y2 = df_step2['price']
# Тестируем
evaluate_custom_model(X2, y2, base_lr, is_target_logged=False)

R²: 0.9125
CV R²: 0.9068 (+/- 0.0067)
RMSE: 1174.45
MAE: 786.93
MAPE: 43.29%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
          carat   +5167.2150
              y   +2614.1799
clarity_encoded    +814.3987
  color_encoded    +550.2756
    cut_encoded    +145.3579
          depth     +82.6738
          table     -38.6875
              z   -1527.6945
              x   -2129.3540


In [49]:
df_step3 = add_volume_feature(df_step2)

X3 = df_step3[['carat', 'depth', 'table', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y3 = df_step3['price']
# Тестируем
evaluate_custom_model(X3, y3, base_lr, is_target_logged=False)

R² (в долларах): 0.9093
CV R² (в долларах): 0.9044 (+/- 0.0053)
RMSE: 1195.50
MAE: 835.62
MAPE: 48.20%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
         volume   +3141.8159
          carat   +1015.6602
clarity_encoded    +856.4169
  color_encoded    +542.3176
    cut_encoded    +126.6622
          depth      +4.5633
          table      -9.9753


In [48]:
X4 = df_step3[['carat', 'depth', 'table', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y4 = np.log1p(df_step3['price']) # Берем логарифм от цены

print("=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===")
# Вызываем утилиту и передаем True, так как таргет залогорифмирован
evaluate_custom_model(X4, y4, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===
R² (в долларах): 0.6525
CV R² (в ЛОГАРИФМАХ): 0.8811 (+/- 0.0044)
RMSE: 2340.21
MAE: 1158.91
MAPE: 27.59%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
         volume      +0.7512
          carat      +0.2763
clarity_encoded      +0.1465
  color_encoded      +0.1464
    cut_encoded      +0.0221
          table      +0.0199
          depth      +0.0125


In [52]:
X5 = df_step3[['carat', 'depth', 'table', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y5 = np.log1p(df_step3['price']) # Берем логарифм от цены

print("=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===")
# Вызываем утилиту и передаем True, так как таргет залогорифмирован
evaluate_custom_model(X5, y5, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===
R² (в долларах): 0.6525
CV R² (в ЛОГАРИФМАХ): 0.8811 (+/- 0.0044)
RMSE: 2340.21
MAE: 1158.91
MAPE: 27.59%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
         volume      +0.7512
          carat      +0.2763
clarity_encoded      +0.1465
  color_encoded      +0.1464
    cut_encoded      +0.0221
          table      +0.0199
          depth      +0.0125


In [55]:
# Изменяем список признаков: полностью убираем 'depth' и 'table'
X6 = df_step3[['carat', 'cut_encoded', 'color_encoded', 'clarity_encoded', 'volume']]
# Таргет оставляем залогорифмированным, так как он доказал свою эффективность для MAPE

print("=== МЕТРИКИ ПОСЛЕ УДАЛЕНИЯ DEPTH И TABLE ===")
evaluate_custom_model(X6, y5, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ УДАЛЕНИЯ DEPTH И TABLE ===
R² (в долларах): 0.6509
CV R² (в ЛОГАРИФМАХ): 0.8809 (+/- 0.0044)
RMSE: 2345.68
MAE: 1160.26
MAPE: 27.62%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
         volume      +0.6281
          carat      +0.4015
  color_encoded      +0.1462
clarity_encoded      +0.1455
    cut_encoded      +0.0129


In [56]:
# 1. Склеиваем огранку и цвет в один текстовый признак
df_step3['cut_color'] = df_step3['cut'].astype(str) + '_' + df_step3['color'].astype(str)

# 2. Быстро переводим этот новый признак в числа (кодируем через factorize)
df_step3['cut_color_encoded'] = pd.factorize(df_step3['cut_color'])[0]

# 3. Собираем новый список признаков (добавляем cut_color_encoded)
X7 = df_step3[['carat', 'cut_encoded', 'color_encoded', 'clarity_encoded', 'volume', 'cut_color_encoded']]

print("=== МЕТРИКИ ПОСЛЕ СКЛЕИВАНИЯ CUT + COLOR ===")
evaluate_custom_model(X7, y5, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ СКЛЕИВАНИЯ CUT + COLOR ===
R² (в долларах): 0.6518
CV R² (в ЛОГАРИФМАХ): 0.8810 (+/- 0.0044)
RMSE: 2342.51
MAE: 1158.02
MAPE: 27.61%

Коэффициенты модели (после масштабирования):
          Признак  Коэффициент
           volume      +0.6412
            carat      +0.3882
    color_encoded      +0.1469
  clarity_encoded      +0.1450
cut_color_encoded      +0.0134
      cut_encoded      +0.0117


In [58]:
# 1. Бины для карата (проверяем округление до популярных коммерческих пиков)
# Округлим до 1 знака для надежности, чтобы поймать значения около пиков
df_step3['is_carat_pivot'] = df_step3['carat'].round(1).isin([0.3, 0.5, 0.7, 1.0, 1.5, 2.0]).astype(int)

# 2. Interaction term (Признак взаимодействия: Вес * Чистота)
df_step3['carat_clarity_inter'] = df_step3['carat'] * df_step3['clarity_encoded']

# 3. Склеивание Огранки и Чистоты (Cut + Clarity)
df_step3['cut_clarity'] = df_step3['cut'].astype(str) + '_' + df_step3['clarity'].astype(str)
df_step3['cut_clarity_encoded'] = pd.factorize(df_step3['cut_clarity'])[0]

# Собираем все наши новые и старые очищенные признаки вместе
X8 = df_step3[[
    'carat', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded',
    'is_carat_pivot', 'carat_clarity_inter', 'cut_clarity_encoded'
]]

print("=== МЕТРИКИ С ПРОДВИНУТЫМ FEATURE ENGINEERING ===")
evaluate_custom_model(X8, y5, base_lr, is_target_logged=True)

=== МЕТРИКИ С ПРОДВИНУТЫМ FEATURE ENGINEERING ===
R² (в долларах): 0.7325
CV R² (в ЛОГАРИФМАХ): 0.9071 (+/- 0.0020)
RMSE: 2053.16
MAE: 1015.57
MAPE: 24.70%

Коэффициенты модели (после масштабирования):
            Признак  Коэффициент
              carat      +0.4157
             volume      +0.3445
carat_clarity_inter      +0.3438
      color_encoded      +0.1432
        cut_encoded      +0.0181
     is_carat_pivot      +0.0161
cut_clarity_encoded      +0.0004
    clarity_encoded      -0.1131


In [60]:
evaluate_custom_model(X8, y5, robust_lr, is_target_logged=True)

R² (в долларах): 0.7325
CV R² (в ЛОГАРИФМАХ): 0.9071 (+/- 0.0020)
RMSE: 2053.16
MAE: 1015.57
MAPE: 24.70%

Коэффициенты модели (после масштабирования):
            Признак  Коэффициент
              carat      +0.5608
             volume      +0.4764
carat_clarity_inter      +0.3851
      color_encoded      +0.2519
        cut_encoded      +0.0325
     is_carat_pivot      +0.0323
cut_clarity_encoded      +0.0005
    clarity_encoded      -0.1373


In [61]:
evaluate_custom_model(X8, y5, base_rf, is_target_logged=True)

R² (в долларах): 0.9805
CV R² (в ЛОГАРИФМАХ): 0.9872 (+/- 0.0002)
RMSE: 554.60
MAE: 288.24
MAPE: 8.53%
